In [ ]:
import os
import pandas as pd
import json
import matplotlib.pyplot as plt

%matplotlib inline

pd.options.display.max_columns = None
pd.options.display.max_colwidth = 60
pd.options.display.float_format = "{:,.3f}".format

from utils import (
    COMPETITOR_ALL_DICT,
    setup_plot_style,
    fetch_files,
    acc_dict_to_dataframe,
    build_color_mapping,
    plot_overall_accuracy_boxplot,
)

setup_plot_style()
%config InlineBackend.figure_format = 'retina'

In [ ]:
dir_path_base = "/home/KeiHiroshima/magmax-masked"

task_seq = "A"
n_splits = 20  # 5 20 50
model = "ViT-B-16"  # __pretrained__laion400m_e31 ViT-L-14
dataset = "CIFAR100"  # ImageNetR CIFAR100
seed_list = [3, 4, 5]
lambda_ = 0.5

dir_name = "merging_target_data_cameraready"
dir_path_shared = f"{dir_path_base}/logs/{model}/sequential_finetuning/class_incremental/{dir_name}/{dataset}-{n_splits}/taskseq_{task_seq}"

suffix = "ablation_cameraready_debug"  # main_results similarity_methods
dir_name_csv = f"processed_{suffix}"
dir_name_fig = f"figs_{suffix}"

competitor_dict = {
    # "finetune": "Baseline",
    # "select_one_task_vector": "Single task vector",
    # "merge_rnd_mix": "Random Mix",
    # "sum": "Average",
    # "ties": "TIES-Merging",
    "merge_max_abs": "MAGMAX",
    "merge_max_abs_masked_with_targetdata-labels": "Tunable MAGMAX (Labels)",
    # "merge_max_abs_masked_with_targetdata-cosine_embedded": "Tunable MAGMAX (Cosine)",
    "merge_max_abs_masked_with_targetdata-ot_embedded": "Tunable MAGMAX (OT)",
    # "merge_max_abs_masked_with_targetdata-mmd_embedded": "Tunable MAGMAX (MMD)",
}

In [ ]:
with open(
    f"{dir_path_base}/configs/target_data_config_split{n_splits}.json",
    "r",
) as f:
    target_data_configs_split = json.load(f)["dataset_configs"]

target_data_config_to_plot = (
    (
        [target_data_configs_split[0]]
        + target_data_configs_split[-2:]
        + target_data_configs_split[1:-2]
    )
    if n_splits == 50
    else target_data_configs_split
)

print(
    "num_task_to_be_fetched",
    [setting["num_task_to_be_fetched"] for setting in target_data_config_to_plot],
)


In [ ]:
acc_mean_df_all = pd.DataFrame(
    index=[setting["num_task_to_be_fetched"] for setting in target_data_config_to_plot],
    columns=[competitor_dict[key] for key in competitor_dict.keys()],
)
acc_std_df_all = pd.DataFrame(
    index=[setting["num_task_to_be_fetched"] for setting in target_data_config_to_plot],
    columns=[competitor_dict[key] for key in competitor_dict.keys()],
)

for config in target_data_config_to_plot:
    num_task_to_be_fetched = config["num_task_to_be_fetched"]
    ratio_task_to_be_fetched = (
        f"_with_{'_'.join([str(r) for r in config['ratio_task_to_be_fetched']])}_ratio"
    )

    targetdata_id = [
        variant["target_id"] for i, variant in enumerate(config["variants"]) if i < 3
    ]
    print(
        f"Processing target data config: num_task_to_be_fetched={num_task_to_be_fetched}, ratio_task_to_be_fetched={ratio_task_to_be_fetched}, targetdata_id={targetdata_id}"
    )

    acc_mean_dict, std_mean_dict = fetch_files(
        targetdata_id, competitor_dict, dir_path_shared, seed_list, lambda_, dir_name
    )
    acc_mean_df = acc_dict_to_dataframe(acc_mean_dict, competitor_dict)
    acc_std_df = acc_dict_to_dataframe(std_mean_dict, competitor_dict)

    acc_mean_df_all.loc[num_task_to_be_fetched] = acc_mean_df.mean(axis=0)
    acc_std_df_all.loc[num_task_to_be_fetched] = acc_std_df.mean(axis=0)

    plot_overall_accuracy_boxplot(
        acc_mean_df,
        f"{dir_path_shared}/{dir_name_fig}",
        f"{num_task_to_be_fetched}_task_selected{ratio_task_to_be_fetched}",
        figsize=(8, 7),
        rotation=20,
        ylabel="Overall Accuracy (%)",
        show_axvline=False,
    )

In [ ]:
print(acc_mean_df_all.index)
acc_mean_df_all

In [ ]:
acc_std_df_all

In [ ]:
plt.close("all")

all_competitors = list(COMPETITOR_ALL_DICT.values())
color_mapping = build_color_mapping()
current_colors = [
    color_mapping[col] for col in acc_mean_df.columns if col in color_mapping
]

log_scale = True

fontsize = 24
marker_size = 14
fig, ax = (
    plt.subplots(1, 1, figsize=(9.5, 6))
    if log_scale
    else plt.subplots(1, 1, figsize=(10, 6))
)

marker_list = ["o", "s", "^", "D"]
for i, col in enumerate(acc_mean_df_all.columns):
    plt.plot(
        acc_mean_df_all.index,
        acc_mean_df_all[col],
        lw=3,
        marker=marker_list[i],
        markersize=marker_size,
        label=col,
        color=current_colors[i],
    )

if log_scale:
    ax.set_xscale("log")
    suffix = "_log_scale"
else:
    suffix = ""

ax.set_xlabel(
    "Number of Tasks in $\mathcal{D}_{\mathrm{tar}}$", fontsize=fontsize * 1.2
)
ax.set_ylabel("Accuracy", fontsize=fontsize * 1.2)
if n_splits == 5:
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=fontsize * 0.95)
else:
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=fontsize * 0.95)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=fontsize * 0.95)

plt.legend(fontsize=fontsize * 0.95, loc="best")
plt.tight_layout()


for spine in ax.spines.values():
    spine.set_linewidth(2)
    spine.set_edgecolor("black")

os.makedirs(f"{dir_path_shared}/{dir_name_fig}", exist_ok=True)
plt.savefig(
    f"{dir_path_shared}/{dir_name_fig}/accuracy_vs_num_tasks{suffix}.pdf",
    bbox_inches="tight",
)
plt.savefig(
    f"{dir_path_shared}/{dir_name_fig}/accuracy_vs_num_tasks{suffix}.png",
    bbox_inches="tight",
)

plt.show()

In [ ]:
# import matplotlib.font_manager as fm
mathcal_D = chr(0x1D49F)
print("\U0001d49f dep")
print(mathcal_D)
